# PMM Dynamic Single-Pair Sweep

**Automated optimization for one exchange and one trading pair**

This notebook:
1. Resolves a single target pair for one connector from MongoDB
2. Runs the same two-phase search used by the multi-pair sweep:
   - Phase 1: Optuna walk-forward optimization
   - Phase 2: Stress-test the top candidates
3. Exports YAML and a markdown report if the result is profitable
4. Displays a compact summary for the requested connector / pair

**Configuration:** Edit the variables in the first code cell, then Run All.


In [1]:
import sys, os, subprocess, time, logging
from datetime import datetime, timezone

PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")
from pmm_lab.optuna.preflight import print_environment, run_preflight
print_environment()

pmm_lab 0.1.0 | NumPy 2.2.6 | Optuna 4.7.0
MONGO_URI      : SET
OPTUNA_STORAGE : SET
Python     : 3.12.13
NumPy      : 2.2.6
Pandas     : 3.0.1
Optuna     : 4.7.0
pmm_lab    : 0.1.0
Storage    : PostgreSQL (SET)
CPU cores  : 32
OMP_NUM_THREADS          : 1
OPENBLAS_NUM_THREADS     : 1
MKL_NUM_THREADS          : 1
NUMEXPR_NUM_THREADS      : 1


## 1. Configuration

Edit these variables to control the single-pair sweep. Then **Run All** cells below.


In [2]:
# ==============================================================
# SWEEP CONFIGURATION — edit these, then Run All
# ==============================================================
# Use this as the operating rule:
# 3000–5000: coarse screening / pair triage
# 8000–10000: good default for a serious single-pair search in this notebook
# 12000–15000: only for finalists or very noisy pairs
# ==============================================================

CONNECTOR = "mexc"             # Exchange connector to sweep
TRADING_PAIR = "TON-USDT"      # Accepts XMR-USDT or XMR/USDT
QUOTE_ASSET = "USDT"           # Quote asset filter (kept for parity with the base notebook)
N_TRIALS = 12000               # Optuna trials for the target pair
PERC_TRIALS_TEST = .05         # what percentage of N_TRIALS should be completely random
TOP_N = 75                     # Top candidates to stress test
MIN_ROBUST_SCORE = 0.0         # Minimum robust score to export (0 = breakeven)
N_JOBS = 8                     # Parallel Optuna workers

# Preferred interval per connector (add your own)
CONNECTOR_INTERVALS = {
    "nonkyc": "5m",
    "mexc": "5m",
}

# Minimum data requirement (days)
MIN_DATA_DAYS = 28

# Maximum training window (days). Only the most recent N days of candle
# data will be used for walk-forward optimization. Set to None to use all
# available data (original behaviour).
MAX_TRAINING_DAYS = 180

# Feature computation mode for search AND stress/validation.
# False = fast vectorized (for broad search), True = controller-equivalent sliding window.
# NOTE: stress/validation uses the same controller_compat setting as search.
# The pipeline does not yet support split modes (search=False, validation=True)
# at the notebook level. Use the full pipeline's search_controller_compat /
# validation_controller_compat parameters for split-mode runs.
SEARCH_CONTROLLER_COMPAT = False

# Stale data gate — skip pairs whose most recent candle is older than this
MAX_STALE_DAYS = 7

# Phase-1 minimum score to proceed to stress testing
# If best phase-1 score <= this, skip stress (saves compute on clearly bad pairs)
MIN_PHASE1_BEST_FOR_STRESS = 0.0

# ==============================================================

TARGET_TRADING_PAIR = TRADING_PAIR.strip().upper().replace("/", "-")
TARGET_QUOTE_ASSET = TARGET_TRADING_PAIR.split("-")[-1]
if QUOTE_ASSET.upper() != TARGET_QUOTE_ASSET:
    print(f"WARNING: QUOTE_ASSET={QUOTE_ASSET} does not match TRADING_PAIR quote {TARGET_QUOTE_ASSET}; using {TARGET_QUOTE_ASSET}")
    QUOTE_ASSET = TARGET_QUOTE_ASSET

INTERVAL = CONNECTOR_INTERVALS.get(CONNECTOR, "5m")

from pmm_lab.config.defaults import INTERVAL_SECONDS
BAR_INTERVAL_SECONDS = INTERVAL_SECONDS[INTERVAL]

print(f"Connector      : {CONNECTOR}")
print(f"Trading pair   : {TARGET_TRADING_PAIR}")
print(f"Quote asset    : {QUOTE_ASSET}")
print(f"Interval       : {INTERVAL} ({BAR_INTERVAL_SECONDS}s/bar)")
print(f"Trials/pair    : {N_TRIALS}")
print(f"Top-N stress   : {TOP_N}")
print(f"Min score      : {MIN_ROBUST_SCORE}")
print(f"Min data days  : {MIN_DATA_DAYS}")
print(f"Search mode    : controller_compat={SEARCH_CONTROLLER_COMPAT}")
print(f"Max stale days : {MAX_STALE_DAYS}")
print(f"Max training   : {MAX_TRAINING_DAYS}d" if MAX_TRAINING_DAYS else "Max training   : unlimited")


Connector      : mexc
Trading pair   : TON-USDT
Quote asset    : USDT
Interval       : 5m (300s/bar)
Trials/pair    : 12000
Top-N stress   : 75
Min score      : 0.0
Min data days  : 28
Search mode    : controller_compat=False
Max stale days : 7


In [3]:
# ── Preflight: validate storage + worker configuration ──
from pmm_lab.optuna.preflight import run_preflight
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

if N_JOBS > 1 and not _is_postgres:
    print(f"WARNING: N_JOBS={N_JOBS} but storage is not PostgreSQL.")
    print(f"  Forcing N_JOBS=1 for SQLite safety.")
    N_JOBS = 1

try:
    preflight_report = run_preflight(
        n_workers=N_JOBS,
        storage_url=_storage_url,
        strict=False,
    )
except Exception as e:
    print(f"Preflight warning: {e}")

print(f"Final N_JOBS   : {N_JOBS}")
print(f"Storage backend: {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")

Preflight: ALL CHECKS PASSED
Final N_JOBS   : 8
Storage backend: PostgreSQL


## 2. Resolve Target Pair

In [4]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=CONNECTOR, quote_asset=QUOTE_ASSET)

now_ts = datetime.now(timezone.utc).timestamp()

# Compute the training-window cutoff: only use candles from the most recent N days
if MAX_TRAINING_DAYS is not None:
    training_cutoff_ts = now_ts - (MAX_TRAINING_DAYS * 86400)
else:
    training_cutoff_ts = None

# Filter to our selected interval and target pair
candidates = []
stale_exclusions = []
insufficient_exclusions = []

for combo in all_combos:
    if combo["trading_pair"] != TARGET_TRADING_PAIR:
        continue
    if combo["interval"] != INTERVAL:
        continue

    # Cap effective start to training window
        effective_first_ts = combo["first_ts"]
        if training_cutoff_ts is not None:
            effective_first_ts = max(effective_first_ts, training_cutoff_ts)
        data_days = (combo["last_ts"] - effective_first_ts) / 86400

    if data_days < MIN_DATA_DAYS:
        insufficient_exclusions.append({
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "data_days": data_days,
            "reason": f"< {MIN_DATA_DAYS}d data",
        })
        continue

    # Stale-pair gate: check recency of last candle
    last_age_days = (now_ts - combo["last_ts"]) / 86400
    if last_age_days > MAX_STALE_DAYS:
        last_utc = datetime.fromtimestamp(combo["last_ts"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        stale_exclusions.append({
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "data_days": data_days,
            "last_age_days": last_age_days,
            "last_utc": last_utc,
            "reason": f"stale ({last_age_days:.1f}d old > {MAX_STALE_DAYS}d)",
        })
        continue

    candidates.append({
        "trading_pair": combo["trading_pair"],
        "count": combo["count"],
        "first_ts": effective_first_ts,
                "full_first_ts": combo["first_ts"],
        "last_ts": combo["last_ts"],
        "data_days": data_days,
    })

print(f"\n{'='*60}")
print(f"Requested target: {CONNECTOR} / {TARGET_TRADING_PAIR} / {INTERVAL}")
print(f"{'='*60}")

if candidates:
    print(f"Found {len(candidates)} matching pair(s) with >= {MIN_DATA_DAYS} days of data:")
    for c in candidates:
        print(f"  {c['trading_pair']:15s}  {c['count']:>8,} candles  {c['data_days']:5.1f} days")
else:
    print("No matching target pair passed discovery.")
    print("Check the connector, pair spelling, interval, data freshness, and MongoDB coverage.")

if stale_exclusions:
    print(f"\nExcluded {len(stale_exclusions)} stale target(s) (last candle > {MAX_STALE_DAYS}d old):")
    for ex in stale_exclusions:
        print(f"  {ex['trading_pair']:15s}  last={ex['last_utc']}  age={ex['last_age_days']:.1f}d")

if insufficient_exclusions:
    print(f"\nExcluded {len(insufficient_exclusions)} target(s) with insufficient data:")
    for ex in insufficient_exclusions:
        print(f"  {ex['trading_pair']:15s}  {ex['data_days']:.1f} days")

print(f"\nTotal pairs to optimize: {len(candidates)}")



Requested target: mexc / TON-USDT / 5m
Found 1 matching pair(s) with >= 28 days of data:
  TON-USDT           55,234 candles  191.8 days

Total pairs to optimize: 1


In [5]:
from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.study import create_study
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.callbacks import DegeneracyCheckCallback, TrialLoggingCallback
from pmm_lab.optuna.canonicalizer import canonicalize_params
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.stress_selection import select_best_stressed_candidate
from pmm_lab.objective.walkforward import run_walk_forward
from pmm_lab.objective.objective import REJECT_SCORE, objective_v1
from pmm_lab.export.hb_yaml import export_yaml, ExportParams
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.sim.runner import CandleSimRunner

# Preload stress scenarios once (Task 4.1)
stress_scenarios = load_stress_scenarios()

rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

for pair_idx, pair_info in enumerate(candidates):
    pair = pair_info["trading_pair"]
    print(f"\n{'\u2550'*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {CONNECTOR} / {pair} / {INTERVAL}")
    print(f"{'\u2550'*60}")

    pair_start = time.time()

    # ── Load candles ──
    try:
        _start_ts = int(pair_info["first_ts"]) if MAX_TRAINING_DAYS is not None else None
        query = DataQuery(connector=CONNECTOR, trading_pair=pair, interval=INTERVAL, start_ts=_start_ts)
        candles = loader.load_range(query)
        audit = validate_candles(candles, interval=INTERVAL, strict=True)
        if not audit.passed_strict:
            print(f"  SKIP: audit failed — {audit.failure_reasons}")
            sweep_results.append({"pair": pair, "status": "audit_fail", "robust_score": None})
            continue
        dataset_hash = hash_candles(candles)
    except Exception as e:
        print(f"  SKIP: load failed — {e}")
        sweep_results.append({"pair": pair, "status": "load_fail", "robust_score": None})
        continue

    # ── Exchange rules ──
    try:
        pair_rules = resolve_pair_rules(rules_db, CONNECTOR, pair)
    except KeyError:
        # Fall back to connector defaults if pair-specific rules not found
        try:
            pair_rules = resolve_pair_rules(rules_db, CONNECTOR, "DEFAULT")
        except KeyError:
            print(f"  SKIP: no exchange rules for {CONNECTOR}/{pair}")
            sweep_results.append({"pair": pair, "status": "no_rules", "robust_score": None})
            continue

    ref_price = float(np.median(candles["close"]))

    # ── Auto-scale walk-forward windows ──
    dataset_days = len(candles) * BAR_INTERVAL_SECONDS / 86400
    if dataset_days >= 120:
        train_days, test_days, step_days = 42.0, 14.0, 14.0
    elif dataset_days >= 60:
        train_days, test_days, step_days = 21.0, 7.0, 7.0
    elif dataset_days >= 28:
        train_days, test_days, step_days = 10.0, 4.0, 4.0
    else:
        print(f"  SKIP: only {dataset_days:.1f} days of data")
        sweep_results.append({"pair": pair, "status": "insufficient_data", "robust_score": None})
        continue

    print(f"  Candles: {len(candles):,}  Days: {dataset_days:.1f}  "
          f"WF: {train_days}/{test_days}/{step_days}d  Ref: {ref_price:,.4f}")

    # ── Phase 1: Optimization ──
    study_name = f"{CONNECTOR}_{pair}_{INTERVAL}_sweep_v1"

    try:
        study = create_study(
            study_name=study_name,
            storage_url=OPTUNA_STORAGE if OPTUNA_STORAGE else None,
            n_startup_trials=int(N_TRIALS * PERC_TRIALS_TEST),
        )

        objective_fn = create_objective(
            candles=candles,
            pair_rules=pair_rules,
            bar_interval_seconds=BAR_INTERVAL_SECONDS,
            dataset_hash=dataset_hash,
            reference_price=ref_price,
            train_days=train_days,
            test_days=test_days,
            step_days=step_days,
            run_stress=False,
            controller_compat=SEARCH_CONTROLLER_COMPAT,  # Fast mode for broad search
        )

        study.optimize(
            objective_fn,
            n_trials=N_TRIALS,
            callbacks=[DegeneracyCheckCallback()],
            catch=(Exception,),
            n_jobs=N_JOBS,
        )

        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
        ranked = sorted(completed, key=lambda t: t.value, reverse=True)

        if not ranked:
            print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned — NO COMPLETED TRIALS")
            sweep_results.append({"pair": pair, "status": "no_completed_trials", "robust_score": None})
            continue

        best_val = ranked[0].value
        print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned, best={best_val:.4f}")
    except Exception as e:
        print(f"  SKIP: optimization failed — {e}")
        sweep_results.append({"pair": pair, "status": "optim_fail", "robust_score": None})
        continue

    # ── Phase 1 score gate ──
    if best_val <= MIN_PHASE1_BEST_FOR_STRESS:
        print(f"  SKIP STRESS: phase-1 best ({best_val:.4f}) <= {MIN_PHASE1_BEST_FOR_STRESS}")
        sweep_results.append({
            "pair": pair, "status": "phase1_below_threshold",
            "robust_score": best_val, "phase1_best": best_val,
        })
        continue

    # ── Phase 2: Stress top N (with signal cache, dedup, early pruning) ──
    try:
        top_trials = ranked[:min(TOP_N, len(ranked))]

        top_candidates = []
        for trial in top_trials:
            config, reject = canonicalize_params(trial.params, pair_rules, ref_price)
            if config is not None:
                top_candidates.append({
                    "trial_number": trial.number,
                    "phase1_score": trial.value,
                    "params": trial.params,
                    "config": config,
                })

        if not top_candidates:
            print(f"  SKIP: no valid configs to stress test")
            sweep_results.append({"pair": pair, "status": "no_valid_configs", "robust_score": None})
            continue

        # Deduplicate by full config fingerprint (Task 4.4)
        seen_configs = {}
        deduped_candidates = []
        for candidate in top_candidates:
            fingerprint = candidate["config"].to_fingerprint()
            if fingerprint not in seen_configs:
                seen_configs[fingerprint] = True
                deduped_candidates.append(candidate)
        print(f"  Deduped: {len(top_candidates)} -> {len(deduped_candidates)} unique configs")
        top_candidates = deduped_candidates

        # Signal cache + early pruning (Tasks 4.3, 4.5)
        signal_cache = {}
        best, diag = select_best_stressed_candidate(
            top_candidates, candles, pair_rules, BAR_INTERVAL_SECONDS,
            scenarios=stress_scenarios,
            signal_cache=signal_cache,
        )

        if best is None:
            print(f"  SKIP: no candidates survived stress testing")
            sweep_results.append({"pair": pair, "status": "stress_fail", "robust_score": None})
            continue

        best_config = best["config"]
        best_stress = best["stress_report"]
        # Reuse winner baseline metrics (Task 4.2) — no extra sim needed
        bm = best_stress.baseline_metrics

        pair_elapsed = time.time() - pair_start
        print(f"  Best: trial {best['trial_number']}  robust={best['robust_score']:.4f}  "
              f"PnL={bm.pnl_pct:.2f}%  trades={bm.trade_count}  ({pair_elapsed/60:.1f}min)")
        print(f"  Stress diag: evaluated={diag['candidates_evaluated']} "
              f"pruned={diag['candidates_pruned']} "
              f"cache_hits={diag['signal_cache_hits']} misses={diag['signal_cache_misses']}")

    except Exception as e:
        print(f"  SKIP: stress testing failed — {e}")
        sweep_results.append({"pair": pair, "status": "stress_fail", "robust_score": None})
        continue

    # ── Record result ──
    best_metrics = bm
    best_obj = best_stress.baseline_objective
    result_entry = {
        "pair": pair,
        "status": "complete",
        "robust_score": best["robust_score"],
        "baseline_score": best["baseline_score"],
        "worst_score": best["worst_score"],
        "worst_scenario": best["worst_scenario"],
        "pnl_pct": bm.pnl_pct,
        "sharpe": bm.sharpe,
        "max_dd_pct": bm.max_drawdown_pct,
        "trade_count": bm.trade_count,
        "total_fees": bm.total_fees_quote,
        "profit_factor": bm.profit_factor,
        "trial_number": best["trial_number"],
        "best_config": best_config,
        "best_params": best["params"],
        "best_stress": best_stress,
        "dataset_hash": dataset_hash,
        "n_candles": len(candles),
        "dataset_days": dataset_days,
        "train_days": train_days,
        "test_days": test_days,
        "step_days": step_days,
        "study_name": study_name,
    }
    sweep_results.append(result_entry)

    # ── Export if profitable ──
    if best["robust_score"] >= MIN_ROBUST_SCORE:
        try:
            export_params = ExportParams(
                connector_name=CONNECTOR,
                trading_pair=pair,
                candles_connector=CONNECTOR,
                candles_trading_pair=pair,
                interval=INTERVAL,
            )

            yaml_path = export_yaml(
                config=best_config,
                output_path=f"artifacts/sweep/{CONNECTOR}/{pair}_{INTERVAL}_best.yaml",
                export_params=export_params,
                metadata={
                    "dataset_hash": dataset_hash,
                    "trial": best["trial_number"],
                    "phase1_score": best["phase1_score"],
                    "robust_score": best["robust_score"],
                    "worst_scenario": best["worst_scenario"],
                    "worst_score": best["worst_score"],
                    "sweep_date": datetime.now(timezone.utc).isoformat(),
                },
            )

            # Walk-forward for report
            wf_result = run_walk_forward(
                candles=candles, config=best_config, pair_rules=pair_rules,
                bar_interval_seconds=BAR_INTERVAL_SECONDS, dataset_hash=dataset_hash,
                train_days=train_days, test_days=test_days, step_days=step_days,
            )

            checks = run_stop_ship_checks(
                best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
            )

            generate_report(
                study_name=study_name,
                dataset_summary={
                    "connector": CONNECTOR, "trading_pair": pair, "interval": INTERVAL,
                    "n_candles": len(candles), "dataset_hash": dataset_hash,
                    "n_trials_phase1": N_TRIALS, "n_candidates_stressed": len(top_candidates),
                    "total_amount_quote_search_min": 25.0,
                    "total_amount_quote_search_max": 1000.0,
                    "total_amount_quote_ideal": best_config.total_amount_quote,
                    "search_controller_compat": SEARCH_CONTROLLER_COMPAT,
                },
                best_params=best["params"], best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
                stop_ship_checks=checks,
                output_path=f"artifacts/sweep/{CONNECTOR}/{pair}_{INTERVAL}_report.md",
            )

            result_entry["exported"] = True
            result_entry["yaml_path"] = yaml_path
            all_pass = all(checks.values())
            result_entry["all_checks_pass"] = all_pass
            print(f"  EXPORTED  yaml={yaml_path}  checks={'ALL PASS' if all_pass else 'SOME FAIL'}")
        except Exception as e:
            print(f"  Export failed: {e}")
            result_entry["exported"] = False
    else:
        result_entry["exported"] = False
        print(f"  NOT PROFITABLE (robust={best['robust_score']:.4f} < {MIN_ROBUST_SCORE})")

total_elapsed = time.time() - sweep_start
print(f"\n{'\u2550'*60}")
print(f"SWEEP COMPLETE: {len(candidates)} pairs in {total_elapsed/60:.1f} minutes")
print(f"{'\u2550'*60}")


════════════════════════════════════════════════════════════
  [1/1] mexc / TON-USDT / 5m
════════════════════════════════════════════════════════════
  Candles: 55,234  Days: 191.8  WF: 42.0/14.0/14.0d  Ref: 1.6380
  Phase 1: 4592 complete, 7408 pruned, best=135.3361
  Deduped: 75 -> 75 unique configs
  Best: trial 2600  robust=1063.9879  PnL=1778.06%  trades=20696  (384.5min)
  Stress diag: evaluated=75 pruned=71 cache_hits=0 misses=75
  EXPORTED  yaml=artifacts/sweep/mexc/TON-USDT_5m_best.yaml  checks=SOME FAIL

════════════════════════════════════════════════════════════
SWEEP COMPLETE: 1 pairs in 385.0 minutes
════════════════════════════════════════════════════════════


## 3. Sweep: Optimize the Target Pair

For the target pair, the sweep:
1. Loads and validates candles
2. Auto-scales walk-forward windows to fit available data
3. Runs Optuna trials (walk-forward, stress OFF)
4. Stress-tests the top candidates
5. Records the best stress-validated result


## 4. Results Summary

In [6]:
# Print discovery exclusion stats
if stale_exclusions:
    print(f"Stale targets excluded : {len(stale_exclusions)}")
if insufficient_exclusions:
    print(f"Insufficient data      : {len(insufficient_exclusions)}")
print()

# Build summary table
summary_rows = []
for r in sweep_results:
    row = {
        "Pair": r["pair"],
        "Status": r["status"],
    }
    if r["status"] == "complete":
        row.update({
            "Robust": f"{r['robust_score']:.2f}",
            "PnL%": f"{r['pnl_pct']:.2f}",
            "Sharpe": f"{r['sharpe']:.2f}",
            "MaxDD%": f"{r['max_dd_pct']:.2f}",
            "Trades": r["trade_count"],
            "PF": f"{r['profit_factor']:.2f}" if r["profit_factor"] != float('inf') else "∞",
            "Fees": f"{r['total_fees']:.2f}",
            "WorstStress": r.get("worst_scenario", ""),
            "Exported": "✓" if r.get("exported") else "✗",
            "Checks": "PASS" if r.get("all_checks_pass") else "—",
        })
    else:
        row.update({k: "—" for k in ["Robust", "PnL%", "Sharpe", "MaxDD%", "Trades",
                                       "PF", "Fees", "WorstStress", "Exported", "Checks"]})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Sort: completed + exported first, then by robust score
def sort_key(row):
    if row["Status"] != "complete":
        return (2, 0)
    if row["Exported"] == "✓":
        return (0, -float(row["Robust"]))
    return (1, -float(row["Robust"]))

summary_df["_sort"] = summary_df.apply(sort_key, axis=1)
summary_df = summary_df.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

print(f"{'='*60}")
print(f"  SWEEP RESULTS: {CONNECTOR} / {TARGET_TRADING_PAIR} / {INTERVAL}")
print(f"{'='*60}\n")

n_complete = len([r for r in sweep_results if r["status"] == "complete"])
n_exported = len([r for r in sweep_results if r.get("exported")])
n_profitable = len([r for r in sweep_results if r["status"] == "complete" and r["robust_score"] >= MIN_ROBUST_SCORE])

print(f"  Total pairs scanned : {len(candidates)}")
print(f"  Completed           : {n_complete}")
print(f"  Profitable          : {n_profitable} (robust score >= {MIN_ROBUST_SCORE})")
print(f"  Exported            : {n_exported}")
print()

display(summary_df)



  SWEEP RESULTS: mexc / TON-USDT / 5m

  Total pairs scanned : 1
  Completed           : 1
  Profitable          : 1 (robust score >= 0.0)
  Exported            : 1



,Pair,Status,Robust,PnL%,Sharpe,MaxDD%,Trades,PF,Fees,WorstStress,Exported,Checks
0,TON-USDT,complete,1063.99,1778.06,3.33,48.81,20696,1.74,309.34,severe_adverse,✓,—


## 5. Target Pair Detail

In [7]:
profitable = [r for r in sweep_results if r["status"] == "complete"
              and r["robust_score"] is not None and r["robust_score"] >= MIN_ROBUST_SCORE]
profitable.sort(key=lambda r: r["robust_score"], reverse=True)

if not profitable:
    print("No profitable target pair found in this sweep.")
    print(f"Try adjusting MIN_ROBUST_SCORE (currently {MIN_ROBUST_SCORE}) or choosing a different pair.")
else:
    for i, r in enumerate(profitable):
        print(f"\n{'─'*60}")
        print(f"  #{i+1}  {r['pair']}  (robust={r['robust_score']:.4f})")
        print(f"{'─'*60}")
        print(f"  PnL %         : {r['pnl_pct']:.4f}")
        print(f"  Sharpe        : {r['sharpe']:.4f}")
        print(f"  Max DD %      : {r['max_dd_pct']:.4f}")
        print(f"  Trades        : {r['trade_count']}")
        print(f"  Profit Fac.   : {r['profit_factor']:.4f}")
        print(f"  Fees          : {r['total_fees']:.4f}")
        print(f"  Worst stress  : {r['worst_scenario']} ({r['worst_score']:.4f})")
        print(f"  Amount (quote): {r['best_config'].total_amount_quote:.2f}  "
              f"(search range: 25.00 – 1000.00)")
        print(f"  Data          : {r['n_candles']:,} candles, {r['dataset_days']:.1f} days")
        if r.get("yaml_path"):
            print(f"  YAML          : {r['yaml_path']}")
        print(f"  Checks        : {'ALL PASS' if r.get('all_checks_pass') else 'SOME FAILED'}")

    print(f"\n{'='*60}")
    print(f"  {len(profitable)} profitable target pair(s) found!")
    print(f"  Check artifacts/sweep/{CONNECTOR}/ for configs and reports.")
    print(f"{'='*60}")



────────────────────────────────────────────────────────────
  #1  TON-USDT  (robust=1063.9879)
────────────────────────────────────────────────────────────
  PnL %         : 1778.0589
  Sharpe        : 3.3269
  Max DD %      : 48.8133
  Trades        : 20696
  Profit Fac.   : 1.7407
  Fees          : 309.3419
  Worst stress  : severe_adverse (395.5172)
  Amount (quote): 237.24  (search range: 25.00 – 1000.00)
  Data          : 55,234 candles, 191.8 days
  YAML          : artifacts/sweep/mexc/TON-USDT_5m_best.yaml
  Checks        : SOME FAILED

  1 profitable target pair(s) found!
  Check artifacts/sweep/mexc/ for configs and reports.


## 6. Next Steps

For the exported target pair:
1. **Review the report** in `artifacts/sweep/<connector>/`
2. **Verify stop-ship checks** and YAML validation results
3. **Paper trade** using Hummingbot's paper trading mode
4. **Monitor** for at least 1 week before live trading
5. **Compare** live performance to backtest expectations

To re-run for a different target, change `CONNECTOR` and `TRADING_PAIR` in the configuration cell and Run All.
